# 第37课 · DFT 暴力实现 — `X[k]=Σ x[n]e^{-2πikn/N}`，O(N²) 双循环 + numpy 对齐验证

**目标**：实现 O(N²) 朴素 DFT，理解"每个频点（frequency bin） = 信号与一组旋转因子序列的点积"

🔗 Aurora 连接：`src/aurora/audio/transforms.py` → `dft()` 是本节的标准答案

## 复习桥 · L21 的 DFT 矩阵（2 分钟）

L21 已经把 DFT 看成一次矩阵乘法：

```text
X = W @ x
```

其中 `W` 的每一行都是一个频率基向量。L37 不再从矩阵视角出发，
而是把同一件事拆开成循环：

```text
X[k] = Σ x[n] · e^{-2πikn/N}
```

两种写法等价：

- 矩阵版帮你记住：DFT 是线性变换
- 循环版帮你记住：每个 `X[k]` 都是一次“信号 vs 频率基”的点积
- L38-L39 会继续追问：这些点积里有哪些重复计算可以省掉？

如果公式看起来突然变陡，先回看 L21 的 `W @ x`，再回来读本课的 `for k in range(N)`。


In [ ]:
import numpy as np

N = 4
x = np.array([1.0, 0.0, -1.0, 0.0])
k = np.arange(N).reshape(N, 1)
n = np.arange(N).reshape(1, N)
W = np.exp(-2j * np.pi * k * n / N)

X_matrix = W @ x
X_numpy = np.fft.fft(x)

print("W @ x       =", np.round(X_matrix, 4))
print("np.fft.fft =", np.round(X_numpy, 4))
np.testing.assert_allclose(X_matrix, X_numpy, atol=1e-12)
print("✅ L21 矩阵视角与 L37 公式视角完全等价")


← **上一课**　[L36 · 窗函数原理](L36_windows.ipynb)

> 上节课学习了 **窗函数原理**：矩形窗的旁瓣泄漏，Hann / Hamming / Blackman 对比。  
> 本课将探讨 **DFT 暴力实现**。

## 本课剧情：为什么频谱分析不直接"扫频"？

想象你想知道一段音乐里有哪些音符。一种暴力方法：对每个可能的频率（比如 0 到 20000 Hz），分别问"信号里有多少这个频率的成分"——逐一扫描。

这正是 **DFT（离散傅里叶变换）** 的原理，也是 FFT 之前的做法：

$$X[k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-2\pi i k n / N}$$

读法：对第 `k` 个频率，用旋转因子 `e^{-2πikn/N}` 逐点"询问"信号 `x[n]`，再求和——这是信号与第 k 个频率基向量的**点积（dot product）**。

**手算 4 点 DFT**（`x = [1, 0, -1, 0]`，这是一个 k=1 的余弦波）：

```
X[0] = 1 + 0 + (-1) + 0 = 0          ← DC 分量为零
X[1] = 1·e⁰ + 0·e^{-πi/2} + (-1)·e^{-πi} + 0·e^{-3πi/2}
     = 1 + 0 + 1 + 0 = 2             ← k=1 处有信号 ✓
X[2] = 1·e⁰ + 0·e^{-πi} + (-1)·e^{-2πi} + 0·e^{-3πi}
     = 1 + 0 - 1 + 0 = 0
X[3] = ... = 0
```

结论：`X[1]=2` 正确反映了 x[n] 是 k=1 的余弦。

**时间复杂度**：N 个频点 × N 次点积 = O(N²)。N 翻倍时间变 4 倍；FFT 用蝶形分治降到 O(N log N)——这是 L38 的内容。

本节任务：用双循环实现 `naive_dft(x)`，并与 `numpy.fft.fft` 逐点对比验证。

In [ ]:
import numpy as np
import time

## 定位图：N=8 DFT 矩阵全貌

先看这张热力图再推公式。每一行是一个**余弦/正弦基向量**；`X[k]` 就是信号 `x` 与第 `k` 行的点积。矩阵共 N² 次乘法——这正是 O(N²) 的来源，也是 FFT 要消灭的冗余。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

N = 8
k, n = np.mgrid[0:N, 0:N]
F = np.exp(-2j * np.pi * k * n / N)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, data, title, cmap in zip(
    axes, [F.real, F.imag],
    ['Re(F)  余弦基', 'Im(F)  负正弦基'],
    ['RdBu_r', 'PuOr_r'],
):
    im = ax.imshow(data, cmap=cmap, vmin=-1, vmax=1)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('样本 n'); ax.set_ylabel('频点 k')
    ax.set_xticks(range(N)); ax.set_yticks(range(N))
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle('N=8 DFT 矩阵  F[k,n] = exp(−2πi·k·n/8)\n'
             '第 k 行与信号 x 的点积 = X[k]（频点 k 的强度与相位）',
             fontsize=10.5, fontweight='bold')
plt.tight_layout()
plt.show()

## 1. DFT 公式与矩阵形式

单个频点的公式：

```
X[k] = sum_{n=0}^{N-1} x[n] * exp(-2*pi*i*k*n/N)
```

向量化写法：令 `w_k = [exp(-2*pi*i*k*n/N) for n in 0..N-1]`，则 `X[k] = dot(w_k, x)`。

把 k=0..N-1 所有行叠起来就是 **DFT 矩阵** `F`，满足 `X = F @ x`。`F` 是酉矩阵（除以 `sqrt(N)` 后），即 `F_normalized @ F_normalized.conj().T = I`。这意味着 DFT 是一次正交变换（orthogonal transform）——不同频率的基向量彼此垂直。

In [ ]:
# 演示：构造 4×4 DFT 矩阵并验证其酉性
N = 4
k = np.arange(N).reshape(N, 1)   # 列向量
n = np.arange(N).reshape(1, N)   # 行向量
F = np.exp(-2j * np.pi * k * n / N)
F_norm = F / np.sqrt(N)

# 验证酉性：F_norm @ F_norm^H ≈ I
residual = np.abs(F_norm @ F_norm.conj().T - np.eye(N)).max()
print(f"DFT矩阵（N=4）：")
print(np.round(F, 2))
print(f"\n酉性残差（应≈0）：{residual:.2e}")

## 2. 旋转因子 W_N

定义基本旋转因子：

```
W_N = exp(-2*pi*i / N)
```

矩阵元素可以写成：

```
F[k, n] = W_N^{k*n}
```

`W_N^{k*n}` 是单位圆上**逆时针转了 k*n/N 圈**的点。随着 k 增大，每个频率基向量转得更快，对应更高频率的振荡。`W_N` 是 N 次单位根，满足 `W_N^N = 1`，这是 FFT 蝶形运算利用的对称性根源。

In [ ]:
# 演示：N=8 的旋转因子在单位圆上的位置
N = 8
W = np.exp(-2j * np.pi / N)
roots = [W**k for k in range(N)]
print(f"N={N} 的旋转因子（k=0..{N-1}）：")
for k, w in enumerate(roots):
    print(f"  W_8^{k} = {w.real:+.4f} {w.imag:+.4f}i  |模|={abs(w):.4f}")
print(f"\nW_N^N = {W**N:.6f}（应为 1+0i）")

## 3. 频率含义与共轭对称（conjugate symmetry）

频谱 `X[k]` 对应的物理频率（Hz）：

```
f_k = k * sr / N
```

其中 `sr` 是采样率（sample rate，sr）。特殊情况：
- `k=0`：**直流分量（DC component）**，等于信号均值乘以 N
- `k=1..N//2`：正频率（有效频率范围到奈奎斯特 `sr/2`）
- `k=N//2+1..N-1`：负频率（冗余）

实信号必然满足**共轭对称**：

```
X[k] = conj(X[N-k])
```

所以实信号的独立信息只在前 `N//2+1` 个频点，这是后续 RFFT 优化的基础。

In [ ]:
# 演示：直流分量 + 共轭对称
x = np.array([1.0, 2.0, 3.0, 4.0])
X = np.fft.fft(x)
N = len(x)
print(f"x = {x}")
print(f"X = {np.round(X, 4)}")
print(f"\nX[0]（直流）= {X[0].real:.1f}，sum(x) = {sum(x):.1f}")
print(f"\n共轭对称验证：X[k] == conj(X[N-k])")
for k in range(1, N):
    match = np.isclose(X[k], np.conj(X[N-k]))
    print(f"  X[{k}] vs conj(X[{N-k}]): {match}")

## 4. ✏️ 实现 `naive_dft(x)`

**三步实现路线**：

| 步骤 | 操作 | 代码骨架 |
|---|---|---|
| 1 | 预分配输出 | `X = np.zeros(N, dtype=complex)` |
| 2 | 外层 k 循环（N 个频点） | `for k in range(N):` |
| 3 | 内层向量化（N 个时域点） | `n = np.arange(N); X[k] = np.dot(x, np.exp(-2j*np.pi*k*n/N))` |

**验收标准**：`np.allclose(naive_dft(x), np.fft.fft(x))` 通过（atol=1e-10）

**常见错误**：
- 忘了负号：`exp(+2j*...)` 是逆 DFT，不是 DFT
- 实数输入：需先 `x = np.asarray(x, dtype=complex)`（stub 已做）
- 双循环不向量化也可以：`X[k] += x[n] * np.exp(...)` 在 n 上循环

**DFT 物理频率**：`f_k = k × sr / N`，即 k=1 对应 `sr/N` Hz（最低非零频率）。

In [ ]:
def naive_dft(x: np.ndarray) -> np.ndarray:
    """朴素 O(N²) DFT：X[k] = sum_n x[n]*exp(-2*pi*i*k*n/N)"""
    x = np.asarray(x, dtype=complex)
    N = len(x)
    X = np.zeros(N, dtype=complex)
    n = np.arange(N)
    # 删除下方的 raise 行，然后在 for 循环中填入你的实现
    raise NotImplementedError("请实现 naive_dft：参考 DFT 公式，对每个 k 计算旋转因子并求点积")
    for k in range(N):
        # ✏️ TODO: 根据 DFT 定义式，计算频率索引 k 对应的 N 个旋转因子（一个复数序列）
        # ✏️ TODO: 将信号 x 与旋转因子序列求点积，存入 X[k]
        pass
    return X


In [ ]:
# 检查：与 numpy FFT 对比
x = np.array([1.0, 2.0, 3.0, 4.0])
try:
    _out = naive_dft(x)
    _not_impl = False
except NotImplementedError:
    _out = None
    _not_impl = True

if _not_impl or _out is None or not np.any(_out):
    print("⬜ 请先实现 naive_dft 的 TODO 项，再运行此格")
else:
    np.testing.assert_allclose(_out, np.fft.fft(x), atol=1e-10)
    print("✅ naive_dft([1,2,3,4]) 与 np.fft.fft 误差 < 1e-10")
    for N in [8, 16, 32]:
        xr = np.random.randn(N)
        np.testing.assert_allclose(naive_dft(xr), np.fft.fft(xr), atol=1e-10)
    print(f"✅ 随机输入 N=8,16,32 全部通过")


## 5. 参数实验：O(N²) 时间复杂度验证

对 `N = 8, 64, 256` 计时 `naive_dft`：
- N 翻 8 倍 → 时间预期翻 **64 倍**（N² 规律）
- L38 实现 FFT 后，N=256 将从 ~毫秒级降到 ~微秒级（约 1000x 加速）

In [ ]:
# 防呆：stub 未实现时跳过计时，避免产生无意义输出
try:
    _timing_guard = naive_dft(np.array([1.0+0j, 0.0+0j]))
    if not np.any(_timing_guard):
        print("⬜ 请先实现 naive_dft 再运行计时格"); raise SystemExit
except NotImplementedError:
    print("⬜ 请先实现 naive_dft 再运行计时格"); raise SystemExit

results = {}
for N in [8, 64, 256]:
    xr = np.random.randn(N).astype(complex)
    t0 = time.perf_counter()
    for _ in range(20):
        naive_dft(xr)
    elapsed = (time.perf_counter() - t0) / 20 * 1000  # ms
    results[N] = elapsed
    print(f"N={N:4d}: {elapsed:.4f} ms")

# 验证 N²缩放（粗略）
ratio_64_8   = results[64]  / results[8]
ratio_256_64 = results[256] / results[64]
print(f"\n时间比 N=64/N=8   = {ratio_64_8:.1f}x（理论 64x）")
print(f"时间比 N=256/N=64  = {ratio_256_64:.1f}x（理论 16x）")

---
→ **主线顺序**：完成 `naive_dft` 后，先继续 [L38 · FFT 蝶形分治](L38_fft_butterfly.ipynb)，再到 [L39 · 从零手写 FFT](L39_fft_implement.ipynb)。

→ **稍后回看**：完成 L39 后，再打开 [L42 · FFT 图形化](L42_visual_fft.ipynb)，用蝴蝶图与纯音/和弦/噪声的频谱形态检验你的直觉。


## 本课收束

`naive_dft(x)` 将长度 N 的信号映射为 N 个复数频谱系数 `X[k]`，输出的每个分量对应信号在频率 `k*sr/N` Hz 上的幅度与相位。该实现直接对应 Aurora 中 `src/aurora/audio/transforms.py` 的 `dft()` 函数，是后续 L38 FFT（蝶形运算）、L39 FFT 完整实现 和 L43 STFT（短时滑窗）的算法基础。

下一课（L38）把 DFT 的 O(N²) 双循环加速为 O(N log N) 的蝶形分治（butterfly），这就是 FFT 的核心思想。

---
⬇️ **通关检验**：收束小结已读；请完成下方白板挑战后再勾选自评。


## ✏️ 白板挑战：DFT 手算（目标 10 分钟）

盖上屏幕，纸上作答：

**问 1**：`x = [1, 0, -1, 0]`，手算 X[0] 和 X[2]：
- X[0] = x[0]·e^0 + x[1]·e^0 + x[2]·e^0 + x[3]·e^0 = ?（DC 分量）
- X[2] = x[0]·e^0 + x[1]·e^{-πi} + x[2]·e^{-2πi} + x[3]·e^{-3πi} = ?

**问 2**：`x = [1, 1, 1, 1]`（DC 信号），所有 X[k] 应为多少？
- X[0] = ?，X[1] = ?，X[2] = ?，X[3] = ?

**问 3**：N=8，sr=8000 Hz 时，k=3 对应的物理频率是多少 Hz？  
（公式：f_k = k × sr / N）

**问 4**：naive_dft 的时间复杂度是 O(N²)。若 N 从 256 增大到 1024（×4），执行时间变为约多少倍？

**问 5**：X[N-k] = X[k]* （共轭对称）——这对于实信号意味着什么？  
（提示：N=8 时只需计算 X[0]…X[4]，X[5]…X[7] 免费得到）

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：x=[1,0,-1,0]
x = np.array([1.0, 0.0, -1.0, 0.0])
X_np = np.fft.fft(x)
assert np.isclose(X_np[0], 0+0j, atol=1e-12), f"X[0] 应=0，得到 {X_np[0]}"
assert np.isclose(X_np[2], 0+0j, atol=1e-12), f"X[2] 应=0，得到 {X_np[2]}"
try:
    X_mine = naive_dft(x)
    assert np.allclose(X_mine, X_np, atol=1e-10)
    print(f"Q1 ✅  X[0]={X_mine[0]:.4f}，X[2]={X_mine[2]:.4f}（naive_dft 与 numpy 吻合）")
except NotImplementedError:
    print("⬜ Q1：请先实现 naive_dft()，再运行对答案格")

# 问2：x=[1,1,1,1]，DC信号 → X[0]=4，其余=0
x_dc = np.array([1.0, 1.0, 1.0, 1.0])
X_dc = np.fft.fft(x_dc)
assert np.isclose(X_dc[0], 4+0j, atol=1e-12)
assert np.allclose(X_dc[1:], 0+0j, atol=1e-12)
print(f"Q2 ✅  DC信号: X[0]={X_dc[0]:.4f}，X[1:]=≈0（能量全在k=0）")

# 问3：f_k = k*sr/N
k, sr, N = 3, 8000, 8
f_k = k * sr / N
assert f_k == 3000.0
print(f"Q3 ✅  k=3, sr=8000, N=8 → f_3 = {f_k:.0f} Hz")

# 问4：O(N²) 时间复杂度 — N×4 → 时间×16
ratio = (1024**2) / (256**2)
assert ratio == 16.0
print(f"Q4 ✅  N: 256→1024（×4），时间≈ {ratio:.0f}倍（O(N²)）")

# 问5：共轭对称性验证
x_real = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0])
X_real = np.fft.fft(x_real)
N8 = len(x_real)
for k_val in range(1, N8//2):
    assert np.isclose(X_real[k_val], np.conj(X_real[N8-k_val]), atol=1e-10)
print(f"Q5 ✅  实信号共轭对称：X[k]=X[N-k]*，只需计算 X[0..N/2]（节省一半计算量）")
print("\n🎉 DFT 白板挑战通过！X[k]=Σx[n]e^(-2πikn/N) 已内化。")

In [ ]:
# ✏️ 本课自评
l37_review = {
    "dft_formula":            None,  # 记住 X[k]=Σx[n]e^{-2πikn/N}？True/False
    "naive_dft_implemented":  None,  # naive_dft 实现并通过 allclose 验证？True/False
    "physical_frequency":     None,  # 理解 f_k = k·sr/N？True/False
    "on2_complexity":         None,  # 理解 O(N²) 双循环、知道 FFT 是 O(NlogN)？True/False
    "whiteboard_passed":      None,  # 白板挑战纸上推导完成？True/False
    "l21_matrix_bridge":      None,  # 能说出 L21 的 W @ x 与本课 Σ 公式等价？True/False
}

unfilled = [k for k, v in l37_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l37_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L37 全部通关！进入 L38：把重复的 DFT 计算折成蝶形')

---

→ **下一课**　[L38 · FFT 蝶形分治](L38_fft_butterfly.ipynb)

> 下节课将学习 **FFT 蝶形分治**：偶奇拆分、蝶形运算 E[k]+W^k·O[k]，O(N²)→O(N log N)。